In [8]:
import logic.pulsed.pulse_objects as po
from logic.pulsed.sampling_functions import SamplingFunctions as SF
import time
import matplotlib.pyplot as plt
pjl = pulsedjupyterlogic_AWG #Make sure the Logic is started in the Qudi manager!!!!
from tqdm import tqdm

In [21]:
#measurement parameters
laser_power_voltage = podmrlogic.laser_power_voltage

#Driving information for the first LO (SGS)
target_freq_0 = 1.0336e9 
LO_freq_0 = target_freq_0 + 100e6
pi_pulse = 90e-9
power_start = -25
power_stop = -10
power_step = 0.1

tau_start = 0
tau_stop = 600e-9
tau_num = 60
max_sweeps = 150000

#additional information for save tag
tip_name = 'A-F17-26'
sample = 'NbSe2_S6'
temperature = '2.5K'
b_field = '70mT_Bnv'
contact = 'OOC'
extra = ''

In [23]:
power_arr = np.arange(power_start, power_stop, power_step)
tau_arr = np.linspace(tau_start, tau_stop, num=tau_num)

pulsed_sig = np.zeros((len(power_arr),len(tau_arr)))
pulsed_sig_err = np.zeros((len(power_arr),len(tau_arr)))
rabi_period = np.zeros_like(power_arr)
rabi_period_err = np.zeros_like(power_arr)

start_time = datetime.datetime.now()

meas_dict = {'params': {'Parameters for': 'Rabi with MW power sweep for MW source calibration'}}
meas_dict['params']['axis name for coord0'] = 'Power'
meas_dict['params']['power_start (dBm)'] = power_start
meas_dict['params']['power_stop (dBm)'] = power_stop
meas_dict['params']['power_step (dBm)'] = power_step

meas_dict['params']['Resonance frequency (Hz)'] = target_freq_0
meas_dict['params']['Pulsed measurement start'] = tau_start
meas_dict['params']['Pulsed measurement stop'] = tau_stop
meas_dict['params']['Pulsed measurement steps (#)'] = tau_num
meas_dict['params']['Pulsed measurement sweeped unit'] = 's'
meas_dict['params']['Measurement runs pulsed measurement (#)'] = max_sweeps

meas_dict['params']['Measurement start'] = start_time.isoformat('_','seconds')
meas_dict['params']['Measurement stop'] = datetime.datetime.now().isoformat('_','seconds')

meas_dict.update({'Rabi_sig': {'data': pulsed_sig}})
meas_dict['Rabi_sig']['data_err'] = pulsed_sig_err
meas_dict['Rabi_sig']['power_arr'] = power_arr
meas_dict['Rabi_sig']['tau_arr'] = tau_arr
meas_dict['Rabi_sig']['params'] = {'power_start': power_start,'power_stop': power_stop}
meas_dict['Rabi_sig']['xy_units'] = 'dBm'
meas_dict['Rabi_sig']['si_units'] = 'a.u.'
meas_dict['Rabi_sig']['nice_name'] = 'Rabi_signal'

meas_dict.update({'Rabi_period': {'data': rabi_period}})
meas_dict['Rabi_period']['data_err'] = rabi_period_err
meas_dict['Rabi_period']['power_arr'] = power_arr
meas_dict['Rabi_period']['params'] = {'power_start': power_start,'power_stop': power_stop}
meas_dict['Rabi_period']['xy_units'] = 'dBm'
meas_dict['Rabi_period']['si_units'] = 's'
meas_dict['Rabi_period']['nice_name'] = 'Rabi_period'

pjl.initialize_ensemble(laser_power_voltage = laser_power_voltage, LO_freq_0=LO_freq_0, target_freq_0=target_freq_0, power_0=power_start)
pjl.Rabi(tau_start, tau_stop, tau_num)
for idx, power in enumerate(tqdm(power_arr)):
    time.sleep(0.2)
    pjl.pulsed_master_AWG.set_ext_microwave_settings(use_ext_microwave=True, 
                                                frequency=LO_freq_0,
                                                power=power)
    time.sleep(0.2)
    pjl.start_measurement(max_sweeps = max_sweeps, measurement_type='Rabi', tip_name=tip_name, sample=sample,
                          temperature=temperature, b_field=b_field, contact=contact, extra=extra+f'_power_{power}dBm')
    pulsedmeasurement_AWG._pa.fit_param_fit_func_ComboBox.setCurrentFit('sin_exp')
    pulsedmeasurement_AWG.fit_clicked()
    time.sleep(0.2)
    pulsedmeasurement_AWG.save_clicked()
    meas_dict['Rabi_sig']['data'][idx] = pulsedmeasurementlogic_AWG.signal_data[1]
    meas_dict['Rabi_sig']['data_err'][idx] = pulsedmeasurementlogic_AWG.measurement_error[1]
    meas_dict['Rabi_period']['data'][idx] = 1/pulsedmeasurementlogic_AWG.fit_result.params['frequency']
    meas_dict['Rabi_period']['data_err'][idx] = pulsedmeasurementlogic_AWG.fit_result.params['frequency'].stderr/(pulsedmeasurementlogic_AWG.fit_result.params['frequency'])**2
    if afm_scanner_logic.jupyter_meas_stop:
        print('Measurement force stopped')
        break
    
string = datetime.datetime.now().isoformat('_','seconds')+'_'+f'Rabi_MW_power_sweep_MW_source_calibration'+'_'+tip_name+'_'+sample+'_'+temperature+'_'+b_field+'_'+contact+'_'+extra
filename = string.replace(':','-')
filepath = savelogic.get_path_for_module('AttoDRY2200_Pi3_SPM\\')

with open(filepath+filename+'.pickle', 'wb') as f:
    pickle.dump(meas_dict, f)
    

100%|#################################################################################| 150/150 [1:08:41<00:00, 27.48s/it]


In [ ]:
filebeginning = '20250817-0055-31_autosave_scan49_Rabi_with_PODMR_2.5K_70mT_Bnv_thin_region_NbSe2_S6_A-F17-26_' #here the beginning of the filename, including the date and sample name, is needed

file_year = filebeginning[0:4]
file_month = filebeginning[4:6]
file_day = filebeginning[6:8]
path = 'G:\\Data\\Qudi_Data\\'+file_year+'\\'+file_month+'\\'+file_year+file_month+file_day+'\\AttoDRY2200_Pi3_SPM\\'
filepath = path + filebeginning

with open(filepath+'.pickle', 'rb') as f:
    qafm_data = pickle.load(f)

In [ ]:
plt.rcParams.update({'font.size': 10})
fig = plt.figure(figsize = (5, 4))